<a href="https://colab.research.google.com/github/ddesiccantt/EECS498-DC/blob/main/A2_Q1_2_Softmax.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import torch

scores = torch.tensor([
    [1., 2., 1.],
    [3., 4., 1.]
])

y = torch.tensor([0, 2])

# 每行减去该行最大分数，防止取指数时数值过大
# 这个操作不会改变 Softmax 的结果
shifted_scores = scores - scores.max(dim=1, keepdim=True).values

exp_scores = shifted_scores.exp()

# 每一行除以自己的指数总和
probs = exp_scores / exp_scores.sum(dim=1, keepdim=True)

print(probs)
# 约 tensor([[0.2119, 0.5761, 0.2119],
#           [0.2595, 0.7054, 0.0351]])

N = scores.shape[0]
rows = torch.arange(N, device=scores.device)

correct_probs = probs[rows, y]
# 约 tensor([0.2119, 0.0351])

sample_losses = -correct_probs.log()
# 约 tensor([1.5514, 3.3490])

loss = sample_losses.mean()

print(loss)
# 约 tensor(2.4502)

tensor([[0.2119, 0.5761, 0.2119],
        [0.2595, 0.7054, 0.0351]])
tensor(2.4502)


In [ ]:
import torch

# ===== 1. 准备数据 =====

# [N, D]：两个样本，每个样本两个特征
X = torch.tensor([
    [1., 2.],
    [3., 4.]
])

# 每个样本的真实类别
y = torch.tensor([0, 2])

# [D, C]：每列对应一个类别的权重
W = torch.tensor([
    [1., 0., -1.],
    [0., 1.,  1.]
])


# ===== 2. 计算 Softmax loss 和梯度 =====

def softmax_loss_and_gradient(X, W, y):
    N = X.shape[0]
    rows = torch.arange(N, device=X.device)

    # 分类分数：[N, C]
    scores = X @ W

    # 每行减去最大值，防止指数溢出
    shifted_scores = scores - scores.max(
        dim=1, keepdim=True
    ).values

    # 转换为概率
    exp_scores = shifted_scores.exp()
    sum_exp = exp_scores.sum(dim=1, keepdim=True)
    probs = exp_scores / sum_exp

    # 稳定地计算 log 概率，再计算平均 loss
    log_probs = shifted_scores - sum_exp.log()
    loss = -log_probs[rows, y].mean()

    # loss 对分数的梯度：真实类别位置减 1
    dscores = probs.clone()
    dscores[rows, y] -= 1
    dscores /= N

    # loss 对权重的梯度
    dW = X.T @ dscores

    return loss, dW


# ===== 3. 训练循环 =====

learning_rate = 0.1
num_steps = 1000

for step in range(num_steps):
    loss, dW = softmax_loss_and_gradient(X, W, y)

    print(f"第 {step + 1} 次更新前，loss = {loss.item():.4f}")

    W = W - learning_rate * dW


# ===== 4. 查看训练结果 =====

final_loss, _ = softmax_loss_and_gradient(X, W, y)
predictions = (X @ W).argmax(dim=1)

print("\n最终 loss：", final_loss.item())
print("预测类别：", predictions)
print("真实类别：", y)
print("训练后的权重：")
print(W)

In [9]:
import torch

# ===== 1. 准备数据 =====

X = torch.tensor([
    [1., 2.],
    [3., 4.]
])

y = torch.tensor([0, 2])

W = torch.tensor([
    [1., 0., -1.],
    [0., 1.,  1.]
])


# ===== 2. Softmax loss 和梯度 =====

def softmax_loss_and_gradient(X, W, y, reg=0.01):
    N = X.shape[0]
    rows = torch.arange(N, device=X.device)

    # 分类分数：[N, C]
    scores = X @ W

    # 每行减去最大值，防止指数溢出
    shifted_scores = scores - scores.max(
        dim=1, keepdim=True
    ).values

    # Softmax 概率
    exp_scores = shifted_scores.exp()
    sum_exp = exp_scores.sum(dim=1, keepdim=True)
    probs = exp_scores / sum_exp

    # 分类损失
    log_probs = shifted_scores - sum_exp.log()
    data_loss = -log_probs[rows, y].mean()

    # L2 正则化损失
    reg_loss = reg * (W ** 2).sum()

    # 总损失
    loss = data_loss + reg_loss

    # 分类损失对分数的梯度
    dscores = probs.clone()
    dscores[rows, y] -= 1
    dscores /= N

    # 总损失对权重的梯度
    dW = X.T @ dscores
    dW += 2 * reg * W

    return loss, dW


# ===== 3. 训练 =====

learning_rate = 0.1
reg = 0.01
num_steps = 1000

for step in range(num_steps):
    loss, dW = softmax_loss_and_gradient(X, W, y, reg)

    W = W - learning_rate * dW

    if step % 100 == 0:
        print(f"第 {step} 次更新前，loss = {loss.item():.4f}")


# ===== 4. 查看结果 =====

final_loss, _ = softmax_loss_and_gradient(X, W, y, reg)

scores = X @ W
predictions = scores.argmax(dim=1)
accuracy = (predictions == y).float().mean()

print("\n最终总 loss：", final_loss.item())
print("预测类别：", predictions)
print("真实类别：", y)
print(f"训练准确率：{accuracy.item():.1%}")
print("训练后的权重：")
print(W)

第 0 次更新前，loss = 2.4902
第 100 次更新前，loss = 0.5985
第 200 次更新前，loss = 0.4480
第 300 次更新前，loss = 0.3920
第 400 次更新前，loss = 0.3699
第 500 次更新前，loss = 0.3606
第 600 次更新前，loss = 0.3566
第 700 次更新前，loss = 0.3547
第 800 次更新前，loss = 0.3539
第 900 次更新前，loss = 0.3534

最终总 loss： 0.3531619906425476
预测类别： tensor([0, 2])
真实类别： tensor([0, 2])
训练准确率：100.0%
训练后的权重：
tensor([[-1.8716, -0.4654,  2.3370],
        [ 1.8371, -0.7585, -0.8085]])
